<a href="https://colab.research.google.com/github/broistg/ML-Assignment-DNAC1/blob/main/notebooks/BTL1_traditional.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 0.&nbsp;User manual

This Google Colab file implements a traditional machine learning pipeline for tabular data (Adult Census Income). To run the entire workflow, simply click Run All (library installation and dataset loading are automated).

# 1.&nbsp;Prepare the necessary data and libraries

## 1.1.&nbsp;Verify and install necessary libraries

In [1]:
# Verify required libraries
import sys
import subprocess

def pip_install(packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + packages)

pip_install(["scikit-learn", "pandas", "matplotlib", "seaborn", "torch", "torchvision", "pytorch-tabnet", "wget"])

# Import libraries
import wget
from pathlib import Path
import os
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_curve, auc, classification_report
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Set seed random
RND = 42
np.random.seed(RND)
import random
random.seed(RND)
torch.manual_seed(RND)

## 1.2.&nbsp;Download and store the dataset

In [2]:
url_train = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
url_test = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test"
dataset_name = 'census-income'
col_names = [
'age','workclass','fnlwgt','education','education-num','marital-status',
'occupation','relationship','race','sex','capital-gain','capital-loss',
'hours-per-week','native-country','income'
]

out_train = Path(os.getcwd()+'/data/'+dataset_name+'.csv')
out_test = Path(os.getcwd()+'/data/'+dataset_name+'_test.csv')
out_train.parent.mkdir(parents=True, exist_ok=True)

if out_train.exists():
    print("Training file already exists.")
else:
    print("Downloading training dataset...")
    wget.download(url_train, out_train.as_posix())
    print("Done!")

if out_test.exists():
    print("Test file already exists.")
else:
    print("Downloading test dataset...")
    wget.download(url_test, out_test.as_posix())
    print("Done!")

dataset_train = pd.read_csv(out_train, header=None, names=col_names, skipinitialspace=True)
print('Training dataset shape:', dataset_train.shape)

# Skip comment first line and remove trailing dots
dataset_test = pd.read_csv(out_test, header=0, names=col_names, skipinitialspace=True, skiprows=1)
dataset_test['income'] = dataset_test['income'].str.replace('.', '', regex=False)
print('Test dataset shape:', dataset_test.shape)

# Combine both into one dataset
dataset = pd.concat([dataset_train, dataset_test], ignore_index=True)
print('Combined dataset shape:', dataset.shape)

Done!
Done!
Training dataset shape: (32561, 15)
Test dataset shape: (16280, 15)
Combined dataset shape: (48841, 15)


# 2.&nbsp;Exploratory Data Analysis (EDA)

In [3]:
# Print the first 5 rows
print(dataset.head())
print(dataset.describe(include='all'))

print('\nValue counts for target (train):')
print(dataset['income'].value_counts())

# Check for missing data (UCI uses '?')
print('\nMissing value counts (including "?" occurrences):')
print((dataset == '?').sum())

# Visualization
try:
  # Distribution of income
  px.histogram(dataset, x='income', color='income', title='Income Distribution').show()

  # Numeric features distribution with boxplots
  numeric_features = ['age','fnlwgt','education-num','capital-gain','capital-loss','hours-per-week']
  for col in numeric_features:
    px.histogram(dataset, x=col, nbins=40, marginal='box', title=f'{col} Distribution').show()

  # Categorical features (top categories)
  categorical_features = [c for c in col_names if c not in numeric_features + ['income']]
  for col in categorical_features:
    vc = dataset[col].value_counts().nlargest(10).index
    subset = dataset[dataset[col].isin(vc)]
    px.histogram(subset, x=col, color='income', barmode='group', title=f'{col} vs Income').update_layout(xaxis={'categoryorder':'total descending'}).show()

  # Correlation heatmap
  corr = dataset[numeric_features].corr()
  px.imshow(corr, text_auto=True, title='Correlation Heatmap (Numeric Features)').show()
except Exception as e:
  print('Plotting skipped due to:', e)

   age         workclass  fnlwgt  education  education-num  \
0   39         State-gov   77516  Bachelors             13   
1   50  Self-emp-not-inc   83311  Bachelors             13   
2   38           Private  215646    HS-grad              9   
3   53           Private  234721       11th              7   
4   28           Private  338409  Bachelors             13   

       marital-status         occupation   relationship   race     sex  \
0       Never-married       Adm-clerical  Not-in-family  White    Male   
1  Married-civ-spouse    Exec-managerial        Husband  White    Male   
2            Divorced  Handlers-cleaners  Not-in-family  White    Male   
3  Married-civ-spouse  Handlers-cleaners        Husband  Black    Male   
4  Married-civ-spouse     Prof-specialty           Wife  Black  Female   

   capital-gain  capital-loss  hours-per-week native-country income  
0          2174             0              40  United-States  <=50K  
1             0             0             

# 3.&nbsp;Data Preprocessing

## 3.1.&nbsp;Data Cleaning and Preparation

In [4]:
# Handle missing values
for col in dataset.select_dtypes(include=['object']).columns:
    dataset[col] = dataset[col].str.strip()

dataset.replace('?', np.nan, inplace=True)

print("\nMissing after cleaning:")
print(dataset.isna().sum())

# Define target and features
target = 'income'
numeric_features = ['age', 'fnlwgt', 'education-num','capital-gain','capital-loss','hours-per-week']
categorical_features = [c for c in col_names if c not in numeric_features + [target]]
print('\nNumeric features:', numeric_features)
print('Categorical features:', categorical_features)
# Convert target to binary (<=50K -> 0, >50K -> 1)
dataset[target] = dataset[target].apply(lambda x: 1 if x == '>50K' else 0)


Missing after cleaning:
age                  0
workclass         2799
fnlwgt               0
education            0
education-num        0
marital-status       0
occupation        2809
relationship         0
race                 0
sex                  0
capital-gain         0
capital-loss         0
hours-per-week       0
native-country     857
income               0
dtype: int64

Numeric features: ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
Categorical features: ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country']


## 3.2.&nbsp;Train/Val/Test split with stratification

In [5]:
# Split dataset into train/val/test sets
train, temp = train_test_split(dataset, test_size=0.3, random_state=RND, stratify=dataset['income'])
val, test = train_test_split(temp, test_size=0.5, random_state=RND, stratify=temp['income'])

print('Original dataset shape:', dataset.shape)
print('Split -> Train:', train.shape, 'Validation:', val.shape, 'Test:', test.shape)

# Keep original train, val, test dataframes for pipeline
X_train = train.drop(columns=[target])
y_train = train[target]

X_val = val.drop(columns=[target])
y_val = val[target]

X_test = test.drop(columns=[target])
y_test = test[target]

print('\nOriginal Train shapes: X:', X_train.shape, 'y:', y_train.shape)
print('Original Validation shapes: X:', X_val.shape, 'y:', y_val.shape)
print('Original Test shapes: X:', X_test.shape, 'y:', y_test.shape)

Original dataset shape: (48841, 15)
Split -> Train: (34188, 15) Validation: (7326, 15) Test: (7327, 15)

Original Train shapes: X: (34188, 14) y: (34188,)
Original Validation shapes: X: (7326, 14) y: (7326,)
Original Test shapes: X: (7327, 14) y: (7327,)


## 3.3.&nbsp;Summary of train/val/test sets

In [6]:
# Create a DataFrame for plotting income distribution across splits
income_counts = pd.DataFrame({
    'Split': ['Train'] * len(y_train) + ['Validation'] * len(y_val) + ['Test'] * len(y_test),
    'Income': pd.concat([y_train, y_val, y_test])
})

# Calculate proportions for plotting
income_proportions = income_counts.groupby('Split')['Income'].value_counts(normalize=True).reset_index(name='Proportion')

# Plotting with Plotly
fig = go.Figure(data=[
    go.Bar(name='<=50K', x=income_proportions[income_proportions['Income'] == 0]['Split'], y=income_proportions[income_proportions['Income'] == 0]['Proportion'], marker_color='skyblue'),
    go.Bar(name='>50K', x=income_proportions[income_proportions['Income'] == 1]['Split'], y=income_proportions[income_proportions['Income'] == 1]['Proportion'], marker_color='salmon')
])

fig.update_layout(barmode='group', title='Income Distribution Across Train, Validation, and Test Sets',
                  xaxis_title='Dataset Split', yaxis_title='Proportion')
fig.show()

## 3.4.&nbsp;Perform imputation, scaling, and encoding

In [7]:
def build_preprocessor(config):
    # Set up scaler
    if config["scaler"]["method"] == 'standard':
        scaler = StandardScaler()
    elif config["scaler"]["method"] == 'minmax':
        scaler = MinMaxScaler(feature_range=config["scaler"]["feature_range"])
    else:
        raise ValueError(f"Unsupported scaler method: {config['scaler']['method']}")

    # Set up numerical imputer
    numeric_imputer = config["imputer"]["numerical"]
    if numeric_imputer["strategy"] == 'constant':
        num_imputer = SimpleImputer(strategy='constant', fill_value=numeric_imputer["fill_value"])
    else:
        num_imputer = SimpleImputer(strategy=numeric_imputer["strategy"])

    numeric_pipeline = Pipeline([
        ('imputer', num_imputer),
        ('scaler', scaler)
    ])

    # Set up categorical imputer
    categorical_imputer = config["imputer"]["categorical"]
    if categorical_imputer["strategy"] == 'constant':
        cat_imputer = SimpleImputer(strategy='constant', fill_value=categorical_imputer["fill_value"])
    else:
        cat_imputer = SimpleImputer(strategy=categorical_imputer["strategy"])

    # Set up encoder
    if config["encoder"]["method"] == 'onehot':
        encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    elif config["encoder"]["method"] == 'ordinal':
        encoder = OrdinalEncoder()
    else:
        raise ValueError(f"Unsupported encoder method: {config['encoder']['method']}")

    categorical_pipeline = Pipeline([
        ('imputer', cat_imputer),
        ('onehot', encoder)
    ])

    # Combine numerical and categorical pipelines
    preprocessor = ColumnTransformer([
        ('num', numeric_pipeline, numeric_features),
        ('cat', categorical_pipeline, categorical_features)
    ])
    return preprocessor

# 4.&nbsp;Traditional Machine Learning Pipeline

## 4.1.&nbsp;Training and Evaluation

In [8]:
def train_and_test(config, X_train, y_train, X_val, y_val, X_test, y_test):
    preprocessor = build_preprocessor(config)

    model = globals()[config["model"]["name"]]
    params = config["model"]["params"]
    classifier = model(**params)

    # Set up pipeline: preprocessor + PCA + classifier
    variance_ratio = float(config["pca"]["variance_ratio"])
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('pca', PCA(n_components=variance_ratio)),
        ('classifier', classifier)
    ])

    # Train the model
    pipe.fit(X_train, y_train)

    # Evaluate on validation set
    y_pred = pipe.predict(X_val)
    print(f"[Validation] {config['model']['name']}, PCA: {variance_ratio}")
    print(classification_report(y_val, y_pred, target_names=['<=50K', '>50K'], digits=5))

    # Evaluate on test set
    y_pred = pipe.predict(X_test)
    print(f"\n[Test] {config['model']['name']}, PCA: {variance_ratio}")
    print(classification_report(y_test, y_pred, target_names=['<=50K', '>50K'], digits=5))

    # Compute and plot ROC curve
    if hasattr(pipe, "predict_proba"):
        y_score = pipe.predict_proba(X_test)[:, 1]
    else:
        y_score = pipe.decision_function(X_test)
    fpr, tpr, thr = roc_curve(y_test, y_score)
    roc_auc = auc(fpr, tpr)

    fig = px.line(
        x=fpr, y=tpr,
        title=f"ROC Curve (AUC = {roc_auc:.4f})",
        labels={'x':'False Positive Rate', 'y':'True Positive Rate'}
    )
    fig.add_scatter(x=[0,1], y=[0,1], mode='lines', line=dict(dash='dash', color='red'), name='Random Chance')

    fig.show()

## 4.2.&nbsp;Model comparision

In [9]:
def model_comparision():
    param_grid = {
        'pca': [0.9, 0.95],
        'classifier': [
            (LogisticRegression, {'max_iter': 300}),
            (SVC, {}),
            (DecisionTreeClassifier, {'max_depth': 6}),
            (RandomForestClassifier, {}),
            (KNeighborsClassifier, {'n_neighbors': 13}),
            (GaussianNB, {})
          ]
    }

    config = {
        "imputer": {
            "numerical": {'strategy': 'median'},
            "categorical": {'strategy': 'constant', 'fill_value': 'Missing'}
        },
        "encoder": {"method": "onehot"},
        "scaler": {"method": "standard"},
    }

    results = []
    for clf, clf_params in param_grid['classifier']:
        for params in param_grid['pca']:
            preprocessor = build_preprocessor(config)
            classifier = clf(**clf_params)

            pipe = Pipeline([
                ('preprocessor', preprocessor),
                ('pca', PCA(n_components=params)),
                ('classifier', classifier)
            ])

            pipe.fit(X_train, y_train)
            acc = pipe.score(X_test, y_test)

            y_pred = pipe.predict(X_test)

            precision = precision_score(y_test, y_pred)
            recall = recall_score(y_test, y_pred)
            f1 = f1_score(y_test, y_pred)

            results.append({
                "Classifier": clf.__name__,
                "PCA": params,
                "Accuracy": round(acc, 5),
                "Precision": round(precision, 5),
                "Recall": round(recall, 5),
                "F1-Score": round(f1, 5),
            })

    results = pd.DataFrame(results)

    fig = go.Figure(data=[go.Table(
        header=dict(values=list(results.columns),
                    fill_color='darkslateblue', font_color='white', align='center'),
        cells=dict(values=[results[col] for col in results.columns],
                  fill_color='lavender', font_color='black', align='center', height=25)
    )])

    fig.update_layout(title_text="Model Comparison Results", title_x=0.5)
    fig.show()

# 5.&nbsp;Choose model configuration



**1.   Imputer (fill missing values)**
*   **Numerical**: 'median' (middle value), 'mean' (average), 'constant' (set a fixed value, must provide 'fill_value')
*   **Categorical:** 'most_frequent' (most common category), 'constant' (set a fixed category, must provide 'fill_value')

**2.   Encoder (convert categorical features to numbers)**
*   'onehot' (create separate binary column for each category)
*   'ordinal' (assign a unique integer to each category)

**3.   Scaler**
*   'standard' (zero mean, unit variance)
*   'minmax' (scale features to a given range, must provide feature_range)

**4.   Principal Component Analysis (PCA)**
*   Keep the % of variance you want

**5. Model options (common classifiers)**
*   **LogisticRegression:** 'max_iter' , 'class_weight' , 'random_state' , ...
*   **SVC:** 'kernel' , 'C' , 'gamma' , ...
*   **DecisionTreeClassifier:** 'criterion' , 'max_depth' , 'min_samples_split' , 'min_samples_leaf' , ...
*   **RandomForestClassifier:** 'n_estimators' , 'criterion' , 'max_depth' , 'min_samples_split' , 'min_samples_leaf' , ...
*   **KNeighborsClassifier:** 'n_neighbors' , 'weights' , 'metric' , ...
*   **GaussianNB:** 'priors' , 'var_smoothing'.
*   **compare:** compare all models on the test set with PCA set to 0.9 and 0.95

In [10]:
config = {
    "imputer": {
          "numerical": {'strategy': 'median'},
          "categorical": {'strategy': 'constant', 'fill_value': 'Missing'}
    },
    "encoder": {"method": "onehot"},
    "scaler": {"method": "standard"},
    "pca": {"variance_ratio": 0.95},
    "model": {
        "name": "compare",
        "params": {'n_estimators': 200, 'max_depth': 6}
    }
}

if config['model']['name'] == 'compare':
    model_comparision()
else:
    train_and_test(config, X_train, y_train, X_val, y_val, X_test, y_test)
